# 🐑 Sheep Activity Classifier v4

### Cambios respecto a v3
- **417 videos** de train (antes 100) → permite backbone más potente
- **EfficientNet-B2** (9.1M params, input 260px) en lugar de B0 — mejor capacidad sin overfitting con 417 videos
- **`n_frames` = 12** (antes 8) — más contexto temporal con más datos disponibles
- **YOLO padding 40%** (antes 15%) — preserva contexto espacial clave para distinguir Sitting/Standing/Grazing
- **Pseudo-labeling con umbral adaptativo** — si confianza media < 0.6, baja automáticamente el umbral
- **Fine-tuning progresivo en 3 fases** en lugar de 2
- **Entrenamiento final sobre todos los datos** guardando mejor época
- Rutas de Kaggle preservadas exactamente


## 0 · Instalación

In [ ]:
!pip install -q timm>=0.9.0 ultralytics>=8.0.0 einops

## 1 · GPU, imports y configuración

In [ ]:
import os, sys, math, random, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import timm
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as T

assert torch.cuda.is_available(), "⚠️ Activa la GPU en Settings → Accelerator."
DEVICE = torch.device("cuda")
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   PyTorch {torch.__version__} · CUDA {torch.version.cuda}")

In [ ]:
# ── Rutas (idénticas a tu notebook anterior) ───────────────────────────
WORKING_DIR = Path("/kaggle/working")
INPUT_DIR   = Path("/kaggle/input")

TRAIN_VIDEO_DIR = WORKING_DIR / "train"
TEST_VIDEO_DIR  = WORKING_DIR / "test"
LABEL_CSV       = INPUT_DIR / "datasets/jeffreyamc/sheep-labels/train.csv"
MODULES_DIR     = INPUT_DIR / "datasets/jeffreyamc/modules/"

PROCESSED_DIR  = WORKING_DIR / "processed"
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(MODULES_DIR))

print(f"TRAIN_DIR : {TRAIN_VIDEO_DIR}")
print(f"TEST_DIR  : {TEST_VIDEO_DIR}")
print(f"LABEL_CSV : {LABEL_CSV}")
print(f"MODULES   : {MODULES_DIR}")

# ── Hiperparámetros v4 ────────────────────────────────────────────────
CFG = {
    # Preprocesamiento
    "n_frames"          : 12,              # ↑ de 8 → más contexto temporal con 417 videos
    "video_ext"         : ".mov",
    "yolo_padding"      : 0.40,            # ↑ de 0.15 → preserva contexto espacial
    # Modelo — EfficientNet-B2 con 417 videos es el balance óptimo
    "backbone"          : "efficientnet_b2",  # 9.1M params, input nativo 260px
    "img_size"          : 260,             # tamaño nativo de B2
    "num_classes"       : 5,
    "dropout"           : 0.45,            # levemente menor con más datos
    # K-Fold
    "n_folds"           : 5,
    # Fase 1: solo clasificador
    "epochs_phase1"     : 25,
    "lr_phase1"         : 3e-4,
    # Fase 2: fine-tuning últimos 3 bloques
    "epochs_phase2"     : 20,
    "lr_phase2"         : 5e-5,
    "unfreeze_phase2"   : 3,
    # Fase 3: fine-tuning más profundo (solo en modelo final)
    "epochs_phase3"     : 15,
    "lr_phase3"         : 1e-5,
    "unfreeze_phase3"   : 6,
    # Pseudo-labeling
    "pseudo_confidence" : 0.80,
    # Común
    "batch_size"        : 16,
    "weight_decay"      : 0.01,
    "warmup_epochs"     : 3,
    "patience"          : 8,
    "mixup_prob"        : 0.4,
    "mixup_alpha"       : 0.3,
    "max_grad_norm"     : 1.0,
    "label_smoothing"   : 0.1,
    "num_workers"       : 2,
    "seed"              : 42,
    "tta_augments"      : 6,
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG["seed"])
print("\nConfig v4 cargada ✅")
print(f"  Backbone : {CFG['backbone']} (img {CFG['img_size']}px)")
print(f"  n_frames : {CFG['n_frames']}")
print(f"  YOLO pad : {CFG['yolo_padding']*100:.0f}%")

## 2 · Preprocesamiento con YOLO padding=40%

In [ ]:
# Importamos preprocess.py del dataset de módulos
# Pero sobreescribimos crop_sheep para usar el padding configurable
import cv2
from preprocess import load_yolo, extract_uniform_frames, get_sheep_bbox


def crop_sheep_v4(frame: np.ndarray, bbox, crop_size: int, padding: float = 0.40):
    """
    Crop de la oveja con padding configurable.
    padding=0.40 → 40% extra en cada lado para preservar contexto espacial.
    Clave para distinguir Sitting/Standing (patas) y Grazing (cabeza abajo).
    """
    h, w = frame.shape[:2]
    if bbox is None:
        return cv2.resize(frame, (crop_size, crop_size), interpolation=cv2.INTER_AREA)

    x1, y1, x2, y2 = bbox
    pad_x = int((x2 - x1) * padding)
    pad_y = int((y2 - y1) * padding)
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)

    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return cv2.resize(frame, (crop_size, crop_size), interpolation=cv2.INTER_AREA)

    # Hacer cuadrado con padding negro
    ch, cw = crop.shape[:2]
    side   = max(ch, cw)
    square = np.zeros((side, side, 3), dtype=np.uint8)
    y_off  = (side - ch) // 2
    x_off  = (side - cw) // 2
    square[y_off:y_off + ch, x_off:x_off + cw] = crop
    return cv2.resize(square, (crop_size, crop_size), interpolation=cv2.INTER_AREA)


def process_video_v4(video_path, yolo_model, output_dir, n_frames=12,
                     target_height=480, crop_size=260, padding=0.40):
    """Versión v4: usa crop_size=260 (nativo B2) y padding=40%."""
    frames = extract_uniform_frames(video_path, n_frames)
    if not frames:
        return False
    os.makedirs(output_dir, exist_ok=True)
    for i, frame in enumerate(frames):
        bbox = get_sheep_bbox(yolo_model, frame)
        crop = crop_sheep_v4(frame, bbox, crop_size, padding)
        cv2.imwrite(
            os.path.join(output_dir, f"frame_{i:02d}.png"),
            cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
        )
    return True


def preprocess_split(video_dir, split, yolo_model):
    out_root    = PROCESSED_DIR / split
    video_files = sorted(Path(video_dir).glob(f"*{CFG['video_ext']}"))
    if not video_files:
        video_files = sorted(Path(video_dir).glob("*.mp4"))
    print(f"[{split.upper()}] {len(video_files)} videos → {out_root}")
    failed = []
    for vf in tqdm(video_files, desc=split):
        out = out_root / vf.stem
        # Reprocesar si el número de frames cambió (v3→v4: 8→12 frames)
        existing = list(out.glob("*.png")) if out.exists() else []
        if len(existing) == CFG["n_frames"]:
            continue
        ok = process_video_v4(
            str(vf), yolo_model, str(out),
            n_frames=CFG["n_frames"],
            crop_size=CFG["img_size"],
            padding=CFG["yolo_padding"],
        )
        if not ok:
            failed.append(vf.name)
    if failed:
        print(f"  ⚠️  {len(failed)} videos fallaron: {failed[:5]}")
    print(f"  ✅ Listo")

print("✅ Funciones de preprocesamiento v4 definidas")

In [ ]:
yolo = load_yolo("yolov8m.pt")
print("✅ YOLO listo")

# Nota: si ya tienes frames de v3 (8 frames, 224px) se reprocesarán
# automáticamente porque n_frames cambió a 12 y crop_size a 260
preprocess_split(TRAIN_VIDEO_DIR, "train", yolo)
preprocess_split(TEST_VIDEO_DIR,  "test",  yolo)

del yolo
torch.cuda.empty_cache()
print("🧹 YOLO liberado")

## 3 · Modelo EfficientNet-B2

In [ ]:
class TemporalAttentionPool(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 4), nn.Tanh(),
            nn.Linear(embed_dim // 4, 1),
        )
    def forward(self, x):           # x: (B, N, D)
        w = F.softmax(self.attn(x), dim=1)
        return (w * x).sum(dim=1)   # (B, D)


class SheepClassifier(nn.Module):
    """
    EfficientNet-B2 (9.1M params, input 260px)
    + Temporal Attention Pool sobre N frames
    + MLP clasificador

    Por qué B2 con 417 videos:
      - B0 (5.3M) queda algo justo para 417 videos
      - B2 (9.1M) es el punto óptimo entre capacidad y regularización
      - ViT-Small (22M) sería mejor con 600+ videos
    """
    def __init__(self, num_classes=5, n_frames=12,
                 backbone_name="efficientnet_b2",
                 dropout=0.45, unfreeze_last_n=0):
        super().__init__()
        self.n_frames = n_frames
        self.backbone = timm.create_model(
            backbone_name, pretrained=True,
            num_classes=0, global_pool="avg"
        )
        embed_dim = self.backbone.num_features  # 1408 para B2

        self._set_frozen(unfreeze_last_n)
        self.temporal_pool = TemporalAttentionPool(embed_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes),
        )
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def _set_frozen(self, unfreeze_last_n):
        for p in self.backbone.parameters():
            p.requires_grad = False
        if unfreeze_last_n > 0:
            blocks = list(self.backbone.blocks)
            for block in blocks[-unfreeze_last_n:]:
                for p in block.parameters():
                    p.requires_grad = True
            for name in ["conv_head", "bn2"]:
                layer = getattr(self.backbone, name, None)
                if layer:
                    for p in layer.parameters():
                        p.requires_grad = True
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"  Params: {total/1e6:.2f}M total | {trainable/1e6:.2f}M entrenables ({100*trainable/total:.1f}%)")

    def forward(self, clip):        # clip: (B, N, C, H, W)
        B, N, C, H, W = clip.shape
        emb    = self.backbone(clip.view(B * N, C, H, W)).view(B, N, -1)
        pooled = self.temporal_pool(emb)
        return self.classifier(pooled)


class SmoothedCE(nn.Module):
    def __init__(self, smoothing=0.1, num_classes=5):
        super().__init__()
        self.s = smoothing
        self.c = num_classes
    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=-1)
        if targets.dim() == 1:
            oh = torch.zeros_like(log_p).fill_(self.s / (self.c - 1))
            oh.scatter_(1, targets.unsqueeze(1), 1.0 - self.s)
        else:
            oh = targets * (1 - self.s) + self.s / self.c
        return -(oh * log_p).sum(dim=-1).mean()


def build_model(unfreeze=0):
    return SheepClassifier(
        num_classes=CFG["num_classes"],
        n_frames=CFG["n_frames"],
        backbone_name=CFG["backbone"],
        dropout=CFG["dropout"],
        unfreeze_last_n=unfreeze,
    ).to(DEVICE)


# Test rápido
print("Test de arquitectura:")
_m = build_model(0)
_x = torch.randn(2, CFG["n_frames"], 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
with torch.no_grad():
    _o = _m(_x)
print(f"  Input : {list(_x.shape)}")
print(f"  Output: {list(_o.shape)}  (esperado [2, 5])")
del _m, _x, _o
torch.cuda.empty_cache()

## 4 · Dataset con augmentation para 260px

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = CFG["img_size"]  # 260 para EfficientNet-B2


class TemporalConsistentTransform:
    """Misma transformación espacial para todos los frames del clip."""
    def __init__(self, is_train=True):
        self.is_train  = is_train
        self.to_tensor = T.ToTensor()
        self.normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        self.erase     = T.RandomErasing(p=0.3, scale=(0.02, 0.15))

    def __call__(self, frames):
        if self.is_train:
            i, j, h, w = T.RandomResizedCrop.get_params(
                frames[0], scale=(0.70, 1.0), ratio=(0.85, 1.15)
            )
            flip_h  = random.random() < 0.5
            flip_v  = random.random() < 0.1
            angle   = random.uniform(-20, 20)
            bright  = random.uniform(0.55, 1.45)
            contr   = random.uniform(0.55, 1.45)
            sat     = random.uniform(0.65, 1.35)
            hue     = random.uniform(-0.12, 0.12)
            do_gray = random.random() < 0.06
            do_blur = random.random() < 0.35
            blur_s  = random.uniform(0.1, 2.0) if do_blur else None

        out = []
        for f in frames:
            if self.is_train:
                f = TF.resized_crop(f, i, j, h, w, (IMG_SIZE, IMG_SIZE))
                if flip_h: f = TF.hflip(f)
                if flip_v: f = TF.vflip(f)
                f = TF.rotate(f, angle)
                f = TF.adjust_brightness(f, bright)
                f = TF.adjust_contrast(f, contr)
                f = TF.adjust_saturation(f, sat)
                f = TF.adjust_hue(f, hue)
                if do_gray: f = TF.rgb_to_grayscale(f, num_output_channels=3)
                if blur_s:  f = TF.gaussian_blur(f, kernel_size=5, sigma=blur_s)
            else:
                f = f.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            t = self.normalize(self.to_tensor(f))
            if self.is_train: t = self.erase(t)
            out.append(t)
        return torch.stack(out)  # (N, C, H, W)


class SheepDataset(Dataset):
    def __init__(self, processed_dir, labels_df=None, pseudo_df=None,
                 n_frames=12, is_train=True):
        self.root      = Path(processed_dir)
        self.n_frames  = n_frames
        self.transform = TemporalConsistentTransform(is_train)

        if labels_df is not None:
            df = labels_df.copy()
            if pseudo_df is not None:
                df = pd.concat([df, pseudo_df], ignore_index=True)
            self.samples = [
                (str(r["Id"]), int(r["Target"]))
                for _, r in df.iterrows()
                if (self.root / str(r["Id"])).exists()
            ]
        else:
            self.samples = [
                (d.name, -1)
                for d in sorted(self.root.iterdir()) if d.is_dir()
            ]
        print(f"  Dataset: {len(self.samples)} muestras ({'train aug' if is_train else 'val/test'})")

    def _load_frames(self, video_id):
        files  = sorted((self.root / video_id).glob("frame_*.png"))
        frames = [Image.open(f).convert("RGB") for f in files]
        while len(frames) < self.n_frames:
            frames.append(frames[-1] if frames else Image.new("RGB", (IMG_SIZE, IMG_SIZE)))
        if len(frames) > self.n_frames:
            idx    = np.linspace(0, len(frames)-1, self.n_frames, dtype=int)
            frames = [frames[i] for i in idx]
        return frames[:self.n_frames]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        vid_id, label = self.samples[idx]
        clip = self.transform(self._load_frames(vid_id))
        return {"clip": clip, "label": torch.tensor(label, dtype=torch.long), "video_id": vid_id}


def mixup_batch(clips, labels, num_classes, alpha=0.3):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(clips.size(0), device=clips.device)
    mixed = lam * clips + (1 - lam) * clips[idx]
    oh    = torch.zeros(clips.size(0), num_classes, device=clips.device)
    oh.scatter_(1, labels.unsqueeze(1), 1)
    return mixed, lam * oh + (1 - lam) * oh[idx]


def make_sampler(labels):
    counts  = Counter(labels)
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(weights, len(weights), replacement=True)


def compute_class_weights(labels, num_classes=5):
    counts = Counter(labels)
    total  = len(labels)
    w = torch.tensor(
        [total / (num_classes * counts.get(i, 1)) for i in range(num_classes)],
        dtype=torch.float32
    )
    return (w / w.sum() * num_classes).to(DEVICE)


def oversample_df(df, strategy="median"):
    counts   = df["Target"].value_counts()
    target_n = int(counts.median()) if strategy == "median" else int(counts.max())
    parts    = [df]
    for cls, cnt in counts.items():
        if cnt < target_n:
            extra = df[df["Target"] == cls].sample(
                n=target_n - cnt, replace=True, random_state=42
            )
            parts.append(extra)
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("✅ Dataset y transforms definidos")

## 5 · Utilidades de entrenamiento

In [ ]:
def cosine_warmup(optimizer, warmup, total):
    def f(ep):
        if ep < warmup: return (ep + 1) / warmup
        p = (ep - warmup) / max(1, total - warmup)
        return 0.5 * (1 + math.cos(math.pi * p))
    return LambdaLR(optimizer, f)


class EarlyStopping:
    def __init__(self, patience=8, delta=1e-4):
        self.p = patience; self.d = delta
        self.c = 0; self.best = None
    def step(self, s):
        if self.best is None or s > self.best + self.d:
            self.best = s; self.c = 0; return False
        self.c += 1; return self.c >= self.p
    def reset(self):
        self.c = 0; self.best = None


def make_optimizer(model, lr):
    decay, nodecay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (nodecay if ("bias" in n or "norm" in n or "bn" in n) else decay).append(p)
    return AdamW(
        [{"params": decay,   "weight_decay": CFG["weight_decay"]},
         {"params": nodecay, "weight_decay": 0.0}],
        lr=lr,
    )


def run_epoch(model, loader, optimizer, criterion, scaler, is_train):
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for batch in tqdm(loader, desc=" Train" if is_train else "  Val ", leave=False):
            clips  = batch["clip"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)

            use_mix = is_train and (np.random.random() < CFG["mixup_prob"])
            if use_mix:
                clips, labels_s = mixup_batch(clips, labels, CFG["num_classes"], CFG["mixup_alpha"])

            if is_train: optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                logits = model(clips)
                loss   = criterion(logits, labels_s if use_mix else labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(batch["label"].numpy())

    f1_pc = f1_score(all_labels, all_preds, average=None, zero_division=0)
    return {
        "loss"    : total_loss / len(loader),
        "macro_f1": float(f1_pc.mean()),
        "f1_pc"   : f1_pc.tolist(),
    }


def train_fold(train_df, val_df, fold_n, pseudo_df=None):
    """
    Entrena un fold en 2 fases.
    Fase 1: solo clasificador (backbone congelado)
    Fase 2: fine-tuning de los últimos N bloques
    """
    tag = f"fold{fold_n}"
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold_n}  |  Train: {len(train_df)}  |  Val: {len(val_df)}")
    if pseudo_df is not None and len(pseudo_df) > 0:
        print(f"  + {len(pseudo_df)} pseudo-labels del test")
    print(f"{'─'*60}")

    train_os = oversample_df(train_df)
    train_ds = SheepDataset(
        str(PROCESSED_DIR / "train"), train_os,
        pseudo_df=pseudo_df, n_frames=CFG["n_frames"], is_train=True
    )
    val_ds = SheepDataset(
        str(PROCESSED_DIR / "train"), val_df,
        n_frames=CFG["n_frames"], is_train=False
    )

    sampler      = make_sampler(train_os["Target"].tolist())
    train_loader = DataLoader(
        train_ds, batch_size=CFG["batch_size"], sampler=sampler,
        num_workers=CFG["num_workers"], pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG["batch_size"], shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=True
    )

    best_f1   = 0.0
    ckpt_path = CHECKPOINT_DIR / f"{tag}.pt"
    history   = []

    phases = [
        (1, CFG["epochs_phase1"], CFG["lr_phase1"], 0),
        (2, CFG["epochs_phase2"], CFG["lr_phase2"], CFG["unfreeze_phase2"]),
    ]

    for phase, n_ep, lr, unfreeze in phases:
        print(f"\n  ── Fase {phase}: lr={lr:.0e} | unfreeze={unfreeze} bloques | {n_ep} épocas ──")

        model = build_model(unfreeze)
        if phase == 2 and ckpt_path.exists():
            state = {k.replace("_orig_mod.", ""): v
                     for k, v in torch.load(ckpt_path, map_location=DEVICE)["model_state"].items()}
            model.load_state_dict(state, strict=False)
            print("  Pesos fase 1 cargados")

        if torch.__version__ >= "2.0.0":
            try: model = torch.compile(model, mode="reduce-overhead")
            except: pass

        criterion = SmoothedCE(CFG["label_smoothing"], CFG["num_classes"])
        optimizer = make_optimizer(model, lr)
        scheduler = cosine_warmup(optimizer, CFG["warmup_epochs"], n_ep)
        scaler    = GradScaler("cuda")
        es        = EarlyStopping(CFG["patience"])

        for ep in range(1, n_ep + 1):
            lr_now = optimizer.param_groups[0]["lr"]
            tr = run_epoch(model, train_loader, optimizer, criterion, scaler, True)
            va = run_epoch(model, val_loader,   optimizer, criterion, scaler, False)
            scheduler.step()

            history.append({"fold": fold_n, "phase": phase, "epoch": ep,
                            "train_f1": tr["macro_f1"], "val_f1": va["macro_f1"],
                            "train_loss": tr["loss"],   "val_loss": va["loss"]})

            tag_b = " ✓ BEST" if va["macro_f1"] > best_f1 else ""
            print(f"  Ph{phase} Ep{ep:3d} | lr={lr_now:.1e} | "
                  f"Tr L={tr['loss']:.3f} F1={tr['macro_f1']:.3f} | "
                  f"Va L={va['loss']:.3f} F1={va['macro_f1']:.3f}{tag_b}")
            print(f"    F1/clase: {[f'{v:.2f}' for v in va['f1_pc']]}")

            if va["macro_f1"] > best_f1:
                best_f1 = va["macro_f1"]
                torch.save({"model_state": model.state_dict(),
                            "val_f1": best_f1, "cfg": CFG}, ckpt_path)
            if es.step(va["macro_f1"]):
                print(f"  ⏹ Early stop fase {phase}, época {ep}")
                break

        del model; torch.cuda.empty_cache()

    return best_f1, str(ckpt_path), history

print("✅ Utilidades definidas")

## 6 · Cargar etiquetas y K-Fold Ronda 1

In [ ]:
df = pd.read_csv(LABEL_CSV)
df.columns = ["Id", "Target"]
print(f"Train: {len(df)} videos")
print(df["Target"].value_counts().sort_index())

skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

fold_results = []
fold_ckpts   = []
all_history  = []

print(f"\n{'='*60}")
print(f"  RONDA 1: K-Fold con {len(df)} videos de train")
print(f"  Backbone: {CFG['backbone']} | n_frames: {CFG['n_frames']}")
print(f"{'='*60}")

for fold, (tr_idx, va_idx) in enumerate(skf.split(df["Id"], df["Target"])):
    set_seed(CFG["seed"] + fold)
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    best_f1, ckpt, history = train_fold(tr_df, va_df, fold + 1)
    fold_results.append(best_f1)
    fold_ckpts.append(ckpt)
    all_history.extend(history)
    print(f"  ✅ Fold {fold+1} → Best Val F1 = {best_f1:.4f}")

print(f"\n{'='*60}")
for i, f1 in enumerate(fold_results): print(f"  Fold {i+1}: {f1:.4f}")
print(f"  Media R1: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"{'='*60}")

## 7 · Pseudo-labeling con umbral adaptativo

In [ ]:
def load_model_for_inference(ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    cfg_  = ckpt.get("cfg", CFG)
    m = SheepClassifier(
        num_classes=cfg_["num_classes"],
        n_frames=cfg_["n_frames"],
        backbone_name=cfg_["backbone"],
        dropout=0.0,
        unfreeze_last_n=0,
    ).to(DEVICE)
    state = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
    m.load_state_dict(state, strict=False)
    m.eval()
    return m


@torch.no_grad()
def ensemble_predict_proba(models, frames_pil, n_tta=4):
    all_p = []
    tf_v  = TemporalConsistentTransform(is_train=False)
    tf_a  = TemporalConsistentTransform(is_train=True)
    for m in models:
        clip = tf_v(frames_pil).unsqueeze(0).to(DEVICE)
        with autocast("cuda"): all_p.append(F.softmax(m(clip), -1).cpu().numpy())
        for _ in range(n_tta - 1):
            clip = tf_a(frames_pil).unsqueeze(0).to(DEVICE)
            with autocast("cuda"): all_p.append(F.softmax(m(clip), -1).cpu().numpy())
    return np.mean(all_p, axis=0)[0]  # (num_classes,)


# Cargar modelos R1
torch.backends.cudnn.benchmark = True
r1_models = [load_model_for_inference(p) for p in fold_ckpts]
print(f"{len(r1_models)} modelos R1 cargados")

test_ds = SheepDataset(
    str(PROCESSED_DIR / "test"), n_frames=CFG["n_frames"], is_train=False
)
print(f"Videos de test: {len(test_ds)}")

# Predecir con TTA reducido para pseudo-labeling (más rápido)
all_test_records = []
for idx in tqdm(range(len(test_ds)), desc="Pseudo-labeling"):
    vid_id = test_ds.samples[idx][0]
    frames = test_ds._load_frames(vid_id)
    probs  = ensemble_predict_proba(r1_models, frames, n_tta=3)
    all_test_records.append({
        "Id": vid_id, "Predicted": int(probs.argmax()),
        "Confidence": float(probs.max())
    })

all_preds_df = pd.DataFrame(all_test_records)
conf_mean    = all_preds_df["Confidence"].mean()
conf_max     = all_preds_df["Confidence"].max()
print(f"\nConfianza media: {conf_mean:.3f} | máxima: {conf_max:.3f}")

# Umbral adaptativo: si la confianza media es baja, bajamos el umbral
if conf_mean >= 0.65:
    threshold = CFG["pseudo_confidence"]   # 0.80
elif conf_mean >= 0.55:
    threshold = 0.70
elif conf_mean >= 0.45:
    threshold = 0.60
else:
    threshold = 0.55  # mínimo útil

print(f"Umbral adaptativo: {threshold} (conf_media={conf_mean:.3f})")

pseudo_df = pd.DataFrame([
    {"Id": r["Id"], "Target": r["Predicted"]}
    for _, r in all_preds_df.iterrows()
    if r["Confidence"] >= threshold
])

print(f"\n✅ Pseudo-labels: {len(pseudo_df)}/{len(all_preds_df)} "
      f"({100*len(pseudo_df)/len(all_preds_df):.0f}%)")
if len(pseudo_df) > 0:
    print(pseudo_df["Target"].value_counts().sort_index())

# Visualizar distribución de confianzas
fig, ax = plt.subplots(figsize=(8, 4))
all_preds_df["Confidence"].hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
ax.axvline(threshold, color="tomato", linestyle="--", label=f"Umbral={threshold}")
ax.set_xlabel("Confianza"); ax.set_title("Distribución de confianza en test (R1 ensemble)")
ax.legend(); plt.tight_layout(); plt.show()

# Guardar predicciones R1
all_preds_df.to_csv(CHECKPOINT_DIR / "r1_test_preds.csv", index=False)

del r1_models; torch.cuda.empty_cache()

## 8 · K-Fold Ronda 2 (train + pseudo-labels)

In [ ]:
# Copiar frames de test → train para pseudo-labels
if len(pseudo_df) > 0:
    copied = 0
    for _, row in pseudo_df.iterrows():
        src = PROCESSED_DIR / "test"  / str(row["Id"])
        dst = PROCESSED_DIR / "train" / str(row["Id"])
        if src.exists() and not dst.exists():
            shutil.copytree(str(src), str(dst))
            copied += 1
    print(f"  {copied} carpetas de pseudo-labels copiadas a processed/train")
else:
    print("  Sin pseudo-labels suficientes — Ronda 2 = Ronda 1 sin pseudo")

print(f"\n{'='*60}")
print(f"  RONDA 2: train ({len(df)}) + pseudo-labels ({len(pseudo_df)})")
print(f"{'='*60}")

r2_results = []
r2_ckpts   = []
r2_history = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(df["Id"], df["Target"])):
    set_seed(CFG["seed"] + fold + 100)
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)

    best_f1, ckpt, history = train_fold(
        tr_df, va_df, fold + 1,
        pseudo_df=pseudo_df if len(pseudo_df) > 0 else None
    )

    new_ckpt = str(CHECKPOINT_DIR / f"r2_fold{fold+1}.pt")
    os.rename(ckpt, new_ckpt)
    r2_results.append(best_f1)
    r2_ckpts.append(new_ckpt)
    r2_history.extend(history)
    print(f"  ✅ R2 Fold {fold+1} → Best Val F1 = {best_f1:.4f}")

print(f"\n{'='*60}")
for i, f1 in enumerate(r2_results): print(f"  R2 Fold {i+1}: {f1:.4f}")
print(f"  Media R1: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"  Media R2: {np.mean(r2_results):.4f} ± {np.std(r2_results):.4f}")
print(f"{'='*60}")

## 9 · Entrenamiento final en todos los datos (3 fases)

In [ ]:
# Mejor ronda
best_round_ckpts = r2_ckpts if np.mean(r2_results) >= np.mean(fold_results) - 0.01 else fold_ckpts
print(f"Mejor ronda: {'R2' if best_round_ckpts == r2_ckpts else 'R1'}")
print(f"  R1 media: {np.mean(fold_results):.4f}")
print(f"  R2 media: {np.mean(r2_results):.4f}")

# Calcular épocas óptimas a partir del historial
hist_all  = pd.DataFrame(all_history + r2_history)
best_eps  = hist_all.loc[hist_all.groupby("fold")["val_f1"].idxmax(), "epoch"]
opt_ep    = max(15, int(best_eps.mean()))
print(f"\nÉpocas óptimas promedio: {opt_ep}")

# Dataset final: train + pseudo-labels con oversample al máximo
full_df = df.copy()
if len(pseudo_df) > 0:
    full_df = pd.concat([full_df, pseudo_df], ignore_index=True)
full_os  = oversample_df(full_df, strategy="max")
print(f"Muestras totales: {len(full_df)} | Tras oversample: {len(full_os)}")
print(full_os["Target"].value_counts().sort_index())

full_ds     = SheepDataset(str(PROCESSED_DIR / "train"), full_os,
                           n_frames=CFG["n_frames"], is_train=True)
full_loader = DataLoader(
    full_ds, batch_size=CFG["batch_size"],
    sampler=make_sampler(full_os["Target"].tolist()),
    num_workers=CFG["num_workers"], pin_memory=True, drop_last=True
)

set_seed(CFG["seed"])
final_model = build_model(unfreeze=0)
criterion   = SmoothedCE(CFG["label_smoothing"], CFG["num_classes"])
best_f1_final = 0.0
FINAL_PATH    = CHECKPOINT_DIR / "final_best.pt"

if torch.__version__ >= "2.0.0":
    try: final_model = torch.compile(final_model, mode="reduce-overhead")
    except: pass

# ── Fase 1: solo clasificador ─────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"  Entrenamiento final Fase 1: {opt_ep} épocas (backbone congelado)")
print(f"{'─'*55}")
optimizer = make_optimizer(final_model, CFG["lr_phase1"])
scheduler = cosine_warmup(optimizer, CFG["warmup_epochs"], opt_ep)
scaler    = GradScaler("cuda")

for ep in range(1, opt_ep + 1):
    m = run_epoch(final_model, full_loader, optimizer, criterion, scaler, True)
    scheduler.step()
    tag = ""
    if m["macro_f1"] > best_f1_final:
        best_f1_final = m["macro_f1"]
        torch.save({"model_state": final_model.state_dict(), "cfg": CFG}, FINAL_PATH)
        tag = "  ✓"
    print(f"  Ep {ep:3d} | Loss={m['loss']:.4f} | F1={m['macro_f1']:.4f}{tag}")

# ── Fase 2: descongelar 3 bloques ────────────────────────────────────
print(f"\n{'─'*55}")
print(f"  Fase 2: fine-tuning {CFG['unfreeze_phase2']} bloques")
print(f"{'─'*55}")

# Recargar mejor de fase 1
state = {k.replace("_orig_mod.", ""): v
         for k, v in torch.load(FINAL_PATH, map_location=DEVICE)["model_state"].items()}
final_model._orig_mod.backbone._set_frozen(CFG["unfreeze_phase2"]) \
    if hasattr(final_model, "_orig_mod") \
    else final_model.backbone._set_frozen(CFG["unfreeze_phase2"])

optimizer2 = make_optimizer(final_model, CFG["lr_phase2"])
scheduler2 = cosine_warmup(optimizer2, 2, CFG["epochs_phase2"])

for ep in range(1, CFG["epochs_phase2"] + 1):
    m = run_epoch(final_model, full_loader, optimizer2, criterion, scaler, True)
    scheduler2.step()
    tag = ""
    if m["macro_f1"] > best_f1_final:
        best_f1_final = m["macro_f1"]
        torch.save({"model_state": final_model.state_dict(), "cfg": CFG}, FINAL_PATH)
        tag = "  ✓"
    print(f"  Ep {ep:3d} | Loss={m['loss']:.4f} | F1={m['macro_f1']:.4f}{tag}")

# ── Fase 3: fine-tuning profundo ─────────────────────────────────────
print(f"\n{'─'*55}")
print(f"  Fase 3: fine-tuning {CFG['unfreeze_phase3']} bloques (LR muy bajo)")
print(f"{'─'*55}")

backbone_obj = final_model._orig_mod.backbone \
    if hasattr(final_model, "_orig_mod") else final_model.backbone
backbone_obj._set_frozen(CFG["unfreeze_phase3"])

optimizer3 = make_optimizer(final_model, CFG["lr_phase3"])
scheduler3 = cosine_warmup(optimizer3, 2, CFG["epochs_phase3"])

for ep in range(1, CFG["epochs_phase3"] + 1):
    m = run_epoch(final_model, full_loader, optimizer3, criterion, scaler, True)
    scheduler3.step()
    tag = ""
    if m["macro_f1"] > best_f1_final:
        best_f1_final = m["macro_f1"]
        torch.save({"model_state": final_model.state_dict(), "cfg": CFG}, FINAL_PATH)
        tag = "  ✓"
    print(f"  Ep {ep:3d} | Loss={m['loss']:.4f} | F1={m['macro_f1']:.4f}{tag}")

print(f"\n💾 Mejor F1 en train: {best_f1_final:.4f}")
print(f"   Guardado: {FINAL_PATH}")

## 10 · Curvas de entrenamiento K-Fold

In [ ]:
hist_df = pd.DataFrame(all_history + r2_history)
hist_df.to_csv(CHECKPOINT_DIR / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ["steelblue", "tomato", "seagreen", "darkorange", "purple"]

for fold_n in range(1, CFG["n_folds"] + 1):
    c = colors[fold_n - 1]
    for rnd, (hist, ls) in enumerate([
        (pd.DataFrame(all_history), "--"),
        (pd.DataFrame(r2_history),  "-"),
    ]):
        fd = hist[hist["fold"] == fold_n]
        if len(fd) == 0: continue
        axes[0].plot(range(len(fd)), fd["val_loss"], color=c, ls=ls, alpha=0.8)
        axes[1].plot(range(len(fd)), fd["val_f1"],   color=c, ls=ls,
                     label=f"F{fold_n} R{'2' if ls=='-' else '1'}")

for ax in axes: ax.set_xlabel("Época"); ax.grid(True, alpha=0.3)
axes[0].set_title("Val Loss (-- R1, — R2)")
axes[1].set_title("Val Macro F1")
axes[1].axhline(np.mean(r2_results), color="black", ls=":",
                label=f"R2 media={np.mean(r2_results):.3f}")
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "curves.png", dpi=120)
plt.show()

## 11 · Inferencia final: ensemble K-Fold + modelo final + TTA

In [ ]:
torch.backends.cudnn.benchmark = True

# Cargar todos los modelos disponibles
all_models = []

# Modelo final (entrenado en todos los datos)
all_models.append(load_model_for_inference(str(FINAL_PATH)))
print(f"✅ Modelo final cargado")

# Mejor ronda de folds
for p in best_round_ckpts:
    all_models.append(load_model_for_inference(p))
print(f"✅ {len(best_round_ckpts)} folds cargados")
print(f"\nEnsemble total: {len(all_models)} modelos")

# Dataset de test
test_ds = SheepDataset(
    str(PROCESSED_DIR / "test"),
    n_frames=CFG["n_frames"], is_train=False
)
print(f"Videos de test: {len(test_ds)}")

# Inferencia con TTA completo
results = []
for idx in tqdm(range(len(test_ds)), desc="Inferencia TTA + Ensemble"):
    vid_id = test_ds.samples[idx][0]
    frames = test_ds._load_frames(vid_id)
    probs  = ensemble_predict_proba(all_models, frames, n_tta=CFG["tta_augments"])
    results.append({"Id": vid_id, "Predicted": int(probs.argmax())})

submission = pd.DataFrame(results)
print(f"\n✅ {len(submission)} predicciones")
print("Distribución:", dict(submission["Predicted"].value_counts().sort_index()))

In [ ]:
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

df_check = pd.read_csv(SUBMISSION_PATH)
assert list(df_check.columns) == ["Id", "Predicted"]
assert df_check["Predicted"].between(0, 4).all()
assert df_check["Id"].nunique() == len(df_check)
assert df_check.isnull().sum().sum() == 0

print(f"✅ Submission válido · {len(df_check)} filas")
print(f"💾 Guardado: {SUBMISSION_PATH}")
submission.head(10)